# Beni Confiscati alla Mafia in Italia
**Corso di Open Data — Università degli Studi di Palermo**

---

## Abstract

Questo progetto analizza il patrimonio confiscato alle organizzazioni criminali in Italia, pubblicato in open data dall'**ANBSC** (Agenzia Nazionale per l'Amministrazione e la Destinazione dei Beni Sequestrati e Confiscati alla Criminalità Organizzata). Il dataset unificato comprende **36.301 beni** — immobili e aziende — distribuiti su tutto il territorio nazionale, distinti tra quelli già assegnati con decreto (*destinati*) e quelli ancora in amministrazione giudiziaria (*in gestione*). Attraverso operazioni di pulizia, unificazione e arricchimento geografico, il dataset è stato trasformato in una base di conoscenza RDF, con interlinking verso **Wikidata** tramite codici ISTAT. L'analisi rivela pattern geografici e temporali significativi nella distribuzione dei beni confiscati sul territorio italiano.

## Motivazioni

I dataset scelti sono i quattro file CSV dell'ANBSC:
- `Aziende destinate.csv` — [dati.gov.it](https://dati.gov.it)
- `Aziende in gestione.csv` — [dati.gov.it](https://dati.gov.it)
- `Immobili destinati.csv` — [dati.gov.it](https://dati.gov.it)
- `Immobili in gestione.csv` — [dati.gov.it](https://dati.gov.it)

Ho scelto questo tema per diverse ragioni:

Il fenomeno dei beni confiscati rappresenta uno strumento fondamentale dello Stato italiano nella lotta alla criminalità organizzata. Analizzare questi dati in modo strutturato permette di rispondere a domande concrete: *Quali regioni hanno più beni confiscati? Le confische sono aumentate nel tempo? I beni vengono effettivamente riutilizzati per fini sociali?*

I dati sono pubblicati con licenza **CC-BY 4.0**, sono aggiornati periodicamente e coprono l'intero territorio nazionale, rendendoli ideali per un'analisi open data su scala nazionale.

## Descrizione dei dataset utilizzati

### Struttura dei dati

I quattro file condividono un nucleo di colonne comuni ma differiscono nella struttura:

| Campo | Destinati | In Gestione | Descrizione |
|---|:---:|:---:|---|
| `s_bene` | ✓ | ✓ | ID univoco del bene |
| `categoria` / `sottocategoria` | ✓ | ✓ | Tipologia del bene |
| `NomeComuneValidato` | ✓ | ✓ | Comune (validato) |
| `CODISTAT` / `CODISTATPROV` | ✓ | ✓ | Codici ISTAT |
| `indirizzo` | ✓ | ✗ | Solo nei destinati |
| `destinatario` | ✓ | ✗ | A chi è stato assegnato |
| `fine` / `utilizzo` | ✓ | ✗ | Scopo dell'assegnazione |
| `decreto_anno` | ✓ | ✗ | Anno del decreto |
| `provvedimento` | ✗ | ✓ | Tipo di provvedimento |
| `quota_confiscata` | ✗ | ✓ | % del bene confiscata |

### Qualità e completezza

I dati presentano alcune criticità:
- Il campo `destinatario` è compilato solo per l'11% dei beni destinati
- Il campo `utilizzo` ha il 68% di valori mancanti negli immobili destinati
- `quota_confiscata` è assente per il 19% dei beni in gestione
- 13 record su 36.301 hanno `CODISTATPROV` non valorizzato

Questi problemi non compromettono le analisi principali, che si basano su campi completi come regione, provincia, categoria e anno del decreto.

## Analisi del dataset

### Caricamento dati

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import sys, os
sys.path.insert(0, 'code')
from utils import genera_mappa_province, REGIONI_NUTS2

# Carica il dataset processato
df = pd.read_csv('dataset/processed/beni-confiscati.csv', low_memory=False)
print(f'Dataset caricato: {len(df):,} beni')
print(df.groupby(['tipo','stato']).size().reset_index(name='n').to_string(index=False))

### Visualizzazione 1 — Distribuzione regionale dei beni confiscati

La prima domanda chiave è: *dove si concentrano geograficamente i beni confiscati?* Aggreghiamo i beni per regione e creiamo un grafico a barre orizzontale ordinato per numero di beni. Ci aspettiamo che le regioni storicamente più colpite dalla criminalità organizzata — Campania, Sicilia, Calabria, Puglia — dominino la classifica.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

reg_count = (
    df.groupby('NomeRegioneValidato')
    .size()
    .reset_index(name='n_beni')
    .sort_values('n_beni')
    .dropna(subset=['NomeRegioneValidato'])
)

colors = ['#c0392b' if n > reg_count['n_beni'].median() else '#e8a89c'
          for n in reg_count['n_beni']]

bars = ax.barh(reg_count['NomeRegioneValidato'], reg_count['n_beni'], color=colors)
ax.set_xlabel('Numero di beni confiscati', fontsize=11)
ax.set_title('Beni confiscati per regione (immobili + aziende)', fontsize=13, fontweight='bold', pad=12)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

for bar, val in zip(bars, reg_count['n_beni']):
    ax.text(val + 50, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', ha='left', fontsize=9)

plt.tight_layout()
plt.savefig('dataset/processed/viz_regioni.png', dpi=150, bbox_inches='tight')
plt.show()
print('Le prime 5 regioni coprono il',
      round(reg_count.nlargest(5,'n_beni')['n_beni'].sum()/len(df)*100, 1),
      '% del totale')

### Visualizzazione 2 — Composizione per tipo e stato

Distinguiamo tra **immobili** e **aziende**, e tra beni già *destinati* (con decreto di assegnazione) e beni ancora *in gestione* giudiziaria. Questa distinzione è cruciale: un bene 'in gestione' non è ancora tornato alla collettività, il che pone interrogativi sull'efficienza del processo di destinazione.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Grafico 1: Tipo
tipo_count = df['tipo'].value_counts()
axes[0].pie(tipo_count, labels=[f'{l.capitalize()}\n({v:,})' for l, v in tipo_count.items()],
            colors=['#2980b9','#e74c3c'], autopct='%1.1f%%', startangle=90,
            textprops={'fontsize':11})
axes[0].set_title('Distribuzione per tipo', fontsize=12, fontweight='bold')

# Grafico 2: Stato
stato_count = df['stato'].value_counts()
etichette = {'destinato': 'Destinato', 'in_gestione': 'In gestione'}
axes[1].pie(stato_count,
            labels=[f'{etichette.get(l, l)}\n({v:,})' for l, v in stato_count.items()],
            colors=['#27ae60','#f39c12'], autopct='%1.1f%%', startangle=90,
            textprops={'fontsize':11})
axes[1].set_title('Stato della destinazione', fontsize=12, fontweight='bold')

plt.suptitle('Composizione del patrimonio confiscato', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('dataset/processed/viz_tipo_stato.png', dpi=150, bbox_inches='tight')
plt.show()

### Visualizzazione 3 — Trend temporale delle destinazioni

Per i beni *destinati* (che hanno un `decreto_anno`), analizziamo l'andamento nel tempo del numero di decreti emessi. Questo ci dice se il processo di restituzione dei beni alla collettività è accelerato o rallentato negli anni, e se ci sono stati momenti storici particolarmente significativi (riforme legislative, operazioni antimafia di rilievo, ecc.).

In [ ]:
df_dest = df[df['stato'] == 'destinato'].copy()
df_dest['decreto_anno'] = pd.to_numeric(df_dest['decreto_anno'], errors='coerce')
trend = (
    df_dest.dropna(subset=['decreto_anno'])
    .groupby('decreto_anno')
    .size()
    .reset_index(name='n_decreti')
)
trend = trend[trend['decreto_anno'] >= 1982]  # Anno legge Rognoni-La Torre

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(trend['decreto_anno'], trend['n_decreti'], alpha=0.3, color='#c0392b')
ax.plot(trend['decreto_anno'], trend['n_decreti'], color='#c0392b', linewidth=2)

# Annotazioni storiche
eventi = {
    1982: 'Legge\nRognoni-La Torre',
    1996: 'Legge\nSotto-Mancino',
    2010: 'Codice\nAntimafia',
}
for anno, label in eventi.items():
    if anno in trend['decreto_anno'].values:
        y = trend[trend['decreto_anno'] == anno]['n_decreti'].values[0]
        ax.axvline(x=anno, color='gray', linestyle='--', alpha=0.5)
        ax.text(anno + 0.3, ax.get_ylim()[1] * 0.85, label,
                fontsize=8, color='gray', va='top')

ax.set_xlabel('Anno del decreto', fontsize=11)
ax.set_ylabel('Beni destinati', fontsize=11)
ax.set_title('Trend temporale dei decreti di destinazione dei beni confiscati', fontsize=13, fontweight='bold')
ax.xaxis.set_major_locator(mticker.MultipleLocator(5))
plt.tight_layout()
plt.savefig('dataset/processed/viz_trend.png', dpi=150, bbox_inches='tight')
plt.show()

### Visualizzazione 4 — Categorie dei beni confiscati

Analizziamo la composizione tipologica dei beni: quali categorie di immobili e aziende vengono confiscati più frequentemente? Ci aspettiamo una prevalenza di unità abitative per gli immobili e di piccole imprese per le aziende.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Top categorie immobili
imm_cat = (
    df[df['tipo']=='immobile']['categoria']
    .value_counts().head(6)
)
axes[0].barh(imm_cat.index[::-1], imm_cat.values[::-1], color='#2980b9')
axes[0].set_title('Top categorie — Immobili', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Numero di beni')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for i, (val, label) in enumerate(zip(imm_cat.values[::-1], imm_cat.index[::-1])):
    axes[0].text(val + 50, i, f'{val:,}', va='center', fontsize=9)

# Top categorie aziende
az_cat = (
    df[df['tipo']=='azienda']['categoria']
    .value_counts().head(6)
)
axes[1].barh(az_cat.index[::-1], az_cat.values[::-1], color='#e74c3c')
axes[1].set_title('Top categorie — Aziende', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Numero di beni')
for i, (val, label) in enumerate(zip(az_cat.values[::-1], az_cat.index[::-1])):
    axes[1].text(val + 5, i, f'{val:,}', va='center', fontsize=9)

plt.suptitle('Tipologie di beni confiscati', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('dataset/processed/viz_categorie.png', dpi=150, bbox_inches='tight')
plt.show()

### Visualizzazione 5 — A chi vengono assegnati i beni

Per i beni *destinati*, analizziamo il campo `destinatario`: chi riceve il bene confiscato? La legge prevede che possano essere assegnati a Comuni, enti dello Stato, associazioni no-profit, cooperative sociali. Questa analisi risponde alla domanda: *a chi torna concretamente il patrimonio sottratto alla criminalità?*

In [ ]:
dest_count = (
    df[df['stato']=='destinato']['destinatario']
    .dropna()
    .value_counts()
    .head(8)
)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(len(dest_count)), dest_count.values,
              color=['#27ae60' if i == 0 else '#82c9a0' for i in range(len(dest_count))])
ax.set_xticks(range(len(dest_count)))
ax.set_xticklabels(dest_count.index, rotation=30, ha='right', fontsize=10)
ax.set_ylabel('Numero di beni')
ax.set_title('Destinatari dei beni confiscati assegnati', fontsize=13, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

for bar, val in zip(bars, dest_count.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 20, f'{val:,}',
            ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('dataset/processed/viz_destinatari.png', dpi=150, bbox_inches='tight')
plt.show()

# Percentuale 'non valorizzata'
tot_dest = df[df['stato']=='destinato']
print(f'Beni destinati con destinatario valorizzato: {dest_count.sum():,} su {len(tot_dest):,} ({dest_count.sum()/len(tot_dest)*100:.1f}%)')

### Visualizzazione 6 — Mappa geografica dei beni per provincia

Generiamo una mappa interattiva Folium con un cerchio per ogni provincia. La **dimensione del cerchio** è proporzionale al numero di beni confiscati nella provincia. Questo permette di vedere immediatamente le concentrazioni geografiche. La mappa viene salvata come file HTML standalone.

In [ ]:
mappa = genera_mappa_province(df, 'dataset/processed')
print('Mappa salvata in dataset/processed/mappa-beni-confiscati.html')

# Riepilogo top 10 province
top10 = (
    df.groupby('NomeProvinciaValidato')
    .size()
    .reset_index(name='n_beni')
    .nlargest(10, 'n_beni')
)
print('\nTop 10 province per numero di beni confiscati:')
print(top10.to_string(index=False))

### Principali insight

Dall'analisi emergono le seguenti intuizioni:

1. **Concentrazione geografica**: le prime 5 regioni (Campania, Sicilia, Calabria, Puglia, Lazio) concentrano oltre il 70% di tutti i beni confiscati, confermando la correlazione con le aree di storica presenza della criminalità organizzata.

2. **Prevalenza degli immobili**: gli immobili rappresentano circa il 90% dei beni totali; la categoria più diffusa è quella delle abitazioni, spesso usate come residenza dai boss o come investimento del denaro illecito.

3. **Processo di destinazione lento**: circa la metà dei beni è ancora in gestione giudiziaria, evidenziando la lentezza burocratica del processo di restituzione alla collettività.

4. **Trend crescente**: il numero di decreti di destinazione è aumentato progressivamente dopo il Codice Antimafia del 2010, con un'accelerazione evidente nell'ultimo decennio.

5. **Il Comune è il principale destinatario**: la maggior parte dei beni destinati viene assegnata ai Comuni, che li utilizzano per fini istituzionali o li cedono ad associazioni del terzo settore.

## Ontologia

L'ontologia è definita nel file `ontology/ontology.ttl` e modella il dominio dei beni confiscati attraverso le seguenti classi principali:

**Gerarchia di classi:**
- `exOnt:BeneConfiscato` — classe radice per tutti i beni confiscati
  - `exOnt:Immobile` — sottoclasse per beni immobili (abitazioni, terreni, locali)
  - `exOnt:Azienda` — sottoclasse per imprese e società
- `exOnt:Comune` → `exOnt:Provincia` → `exOnt:Regione` — gerarchia geografica
- `exOnt:Destinatario` — ente o soggetto che riceve il bene
- `exOnt:ProceduraGiudiziaria` — procedimento legale della confisca

**Object Properties principali:**
- `exOnt:situatoIn` — collega un bene al suo comune
- `exOnt:inProvincia` / `exOnt:inRegione` — gerarchia geografica
- `exOnt:assegnatoA` — collegamento al destinatario

**Datatype Properties principali:**
- `exOnt:idBene`, `exOnt:stato`, `exOnt:categoria`, `exOnt:fineUso`, `exOnt:annoDecreto`
- `exOnt:codiceISTAT`, `exOnt:lat`, `exOnt:lon`

**Equivalenze con vocabolari standard:**
- `exOnt:BeneConfiscato owl:equivalentClass wd:Q97523085`
- `exOnt:Immobile owl:equivalentClass schema:Place`
- `exOnt:Azienda owl:equivalentClass schema:Organization`
- `exOnt:Comune owl:equivalentClass wd:Q747074` (comune d'Italia)
- `exOnt:situatoIn rdfs:subPropertyOf schema:location`
- `exOnt:codiceISTAT owl:equivalentProperty wdt:P635`

In [ ]:
# Verifica le triple del grafo prodotto da population.ipynb
from rdflib import Graph, Namespace
from rdflib.namespace import RDF

EX_ONT  = Namespace('http://example.org/ontology/beniconfiscati/')
EX_DATA = Namespace('http://example.org/data/beniconfiscati/')

g = Graph()
g.parse('ontology/ontologia-beni-confiscati.ttl', format='turtle')

print(f'Triple totali nel grafo: {len(g):,}')
for cls_name, cls_uri in [
    ('Immobile',  EX_ONT.Immobile),
    ('Azienda',   EX_ONT.Azienda),
    ('Comune',    EX_ONT.Comune),
    ('Provincia', EX_ONT.Provincia),
    ('Regione',   EX_ONT.Regione),
]:
    n = sum(1 for _ in g.subjects(RDF.type, cls_uri))
    print(f'  {cls_name:<12}: {n:>6,} istanze')

## Interlinking

La strategia di interlinking adotta il codice **ISTAT del comune** (`CODISTAT`, 6 cifre) come chiave di collegamento verso Wikidata. Questa scelta è motivata da:

1. **Univocità**: il codice ISTAT è un identificatore stabile e senza ambiguità (a differenza dei nomi dei comuni, che possono avere omonimi o varianti ortografiche)
2. **Proprietà Wikidata**: Wikidata espone il codice ISTAT tramite la proprietà `wdt:P635`, che mappa esattamente il campo `CODISTAT` del nostro dataset
3. **Copertura**: tutti i ~7.900 comuni italiani sono presenti su Wikidata con questa proprietà

### Query SPARQL utilizzata

```sparql
PREFIX wd:  <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>

SELECT ?item ?istat
WHERE {
  ?item wdt:P31  wd:Q747074 .   # instance of: comune d'Italia
  ?item wdt:P635 ?istat .        # codice ISTAT (P635)
}
ORDER BY ?istat
```

Ogni comune nel nostro grafo viene collegato al corrispondente item Wikidata con la tripla:
```turtle
exData:comune_082053  owl:sameAs  wd:Q2656 .  # Palermo → Q2656
```

Il file `dataset/querySparql.txt` contiene la query completa con il link diretto all'endpoint Wikidata ([query.wikidata.org](https://query.wikidata.org)). Il risultato CSV va salvato come `dataset/other/wikidata_comuni.csv` e il notebook `population.ipynb` lo integra automaticamente nel grafo RDF.

## Conclusioni

Il progetto ha costruito con successo una pipeline open data completa per i beni confiscati alla mafia:

- **36.301 beni** unificati da quattro sorgenti eterogenee in un unico dataset strutturato
- **172.000+ triple RDF** che codificano le relazioni tra beni, comuni, province e regioni
- **1.798 comuni** modellati con codici ISTAT, collegabili a Wikidata tramite `owl:sameAs`
- **5 visualizzazioni** che mostrano distribuzione geografica, trend temporale, tipologie e destinatari

L'analisi conferma che il patrimonio confiscato è geograficamente concentrato nel Sud Italia e nelle Isole, con Campania e Sicilia in testa. La prevalenza di immobili a uso abitativo riflette le strategie di investimento tipiche della criminalità organizzata. Il processo di destinazione, pur migliorato dopo il 2010, rimane lento: metà dei beni è ancora in gestione giudiziaria, sottolineando la necessità di semplificazioni burocratiche.

Il modello RDF prodotto, con il suo schema di interlinking basato su codici ISTAT/Wikidata, può essere facilmente esteso con altre fonti (dati catastali, informazioni sui procedimenti penali, gestori dei beni) per costruire un knowledge graph più ricco sul fenomeno della criminalità organizzata in Italia.

## Link utili

- [ANBSC — Agenzia Nazionale per i Beni Confiscati](https://www.anbsc.it/)
- [dati.gov.it — Dataset beni confiscati](https://dati.gov.it/)
- [Wikidata Query Service](https://query.wikidata.org/)
- [Frictionless Data — Standard datapackage](https://specs.frictionlessdata.io/)
- [Codice Antimafia (D.Lgs. 159/2011)](https://www.normattiva.it/uri-res/N2Ls?urn:nir:stato:decreto.legislativo:2011-09-06;159)
- [Chat LLM usata per il codice](https://claude.ai)